# Machine Learning Assignment 2

## Breast Cancer Classification

### Dataset Description

The Breast Cancer Wisconsin (Diagnostic) dataset is used for classifying breast tumors as either malignant or benign based on numerical features computed from digitized images of breast mass samples.

The target variable is **diagnosis**, which contains two classes:

- **M** = Malignant
- **B** = Benign

The dataset contains **569 instances** and **30 numerical predictive features**.

In [117]:
import os

print(os.getcwd())
print(os.listdir())

/home/s2026aj05542
['.mozilla', '.bash_logout', '.bash_profile', '.bashrc', '.config', '.cache', 'Desktop', 'Downloads', 'Templates', 'Public', 'Documents', 'Music', 'Pictures', 'Videos', '.local', '.ICEauthority', '.gnupg', '.vnc', '.Xauthority', 'thinclient_drives', '.xsession-errors.old', '.xsession-errors', '.ssh', 'anaconda3', '.conda', '.ipython', '.vscode', '.pki', '.jupyter', '.ipynb_checkpoints', '.keras', '.anaconda', '.wget-hsts', '.continuum', '.condarc', 'anaconda_projects', '.python_history', '.java', '.viminfo', '.idea', '.bash_history', '.xsession', 'workspace', 'data.csv', 'Source_Code.ipynb', 'logistic_model.pkl', 'decision_tree_model.pkl', 'knn_model.pkl', 'naive_bayes_model.pkl', 'svm_model.pkl', 'scaler.pkl', 'test_data.csv']


In [66]:
import pandas as pd
import numpy as np

df = pd.read_csv("data.csv")

df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


## Exploratory Data Analysis

The dataset is examined to understand its dimensions, features, data types, target class distribution, and missing values before preprocessing and model training.

In [68]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

Dataset Shape: (569, 33)

Column Names:
['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32']


In [69]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [70]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
id                           0
diagnosis                    0
radius_mean                  0
texture_mean                 0
perimeter_mean               0
area_mean                    0
smoothness_mean              0
compactness_mean             0
concavity_mean               0
concave points_mean          0
symmetry_mean                0
fractal_dimension_mean       0
radius_se                    0
texture_se                   0
perimeter_se                 0
area_se                      0
smoothness_se                0
compactness_se               0
concavity_se                 0
concave points_se            0
symmetry_se                  0
fractal_dimension_se         0
radius_worst                 0
texture_worst                0
perimeter_worst              0
area_worst                   0
smoothness_worst             0
compactness_worst            0
concavity_worst              0
concave points_worst         0
symmetry_worst               0
fractal_dimension_worst

In [71]:
print("Diagnosis Distribution:")
print(df["diagnosis"].value_counts())

Diagnosis Distribution:
diagnosis
B    357
M    212
Name: count, dtype: int64


## Data Preprocessing

Before training the machine learning models, the dataset is preprocessed to prepare the input features and target variable.

The following preprocessing steps are performed:

1. Remove unnecessary columns that do not contribute to prediction.
2. Convert the categorical target variable `diagnosis` into numerical values:
   - Benign (B) = 0
   - Malignant (M) = 1
3. Separate the independent features (`X`) and target variable (`y`).
4. Split the dataset into training and testing sets.
5. Standardize the features for models that are sensitive to feature scale.

In [73]:
print(df.columns.tolist())

['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32']


In [74]:
df = df.drop(columns=["id", "Unnamed: 32"], errors="ignore")

df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [75]:
df["diagnosis"] = df["diagnosis"].map({"B": 0, "M": 1})

print(df["diagnosis"].value_counts())

diagnosis
0    357
1    212
Name: count, dtype: int64


### Feature and Target Selection

The `diagnosis` column is selected as the target variable. All remaining numerical columns are used as input features for classification.

The target classes are represented as:

- 0 = Benign
- 1 = Malignant

In [77]:
X = df.drop("diagnosis", axis=1)
y = df["diagnosis"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (569, 30)
Target shape: (569,)


### Train-Test Split

The dataset is divided into training and testing sets. 80% of the data is used for training the machine learning models and 20% is reserved for testing.

Stratified sampling is used to maintain the class distribution in both training and testing datasets.

In [79]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (455, 30)
Testing set: (114, 30)


### Feature Scaling

Standardization is applied so that features with different numerical ranges are transformed to a comparable scale.

The scaler is fitted only on the training data and then applied to both the training and testing data. This prevents information from the test dataset from influencing the training process.

The scaled data will be used for models that are sensitive to feature magnitude, such as Logistic Regression, K-Nearest Neighbors, and Support Vector Machine.

In [81]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")

Feature scaling completed.


## Model 1: Logistic Regression

Logistic Regression is a supervised machine learning algorithm used for binary classification. In this dataset, the model classifies breast tumors as Benign (0) or Malignant (1).

The standardized training data is used to train the model, and its performance is evaluated on the test data using the following metrics:

- Accuracy
- AUC
- Precision
- Recall
- F1 Score
- Matthews Correlation Coefficient (MCC)

In [83]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

# Create and train the Logistic Regression model
logistic_model = LogisticRegression(random_state=42)
logistic_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_logistic = logistic_model.predict(X_test_scaled)
y_prob_logistic = logistic_model.predict_proba(X_test_scaled)[:, 1]

# Calculate performance metrics
accuracy_logistic = accuracy_score(y_test, y_pred_logistic)
auc_logistic = roc_auc_score(y_test, y_prob_logistic)
precision_logistic = precision_score(y_test, y_pred_logistic)
recall_logistic = recall_score(y_test, y_pred_logistic)
f1_logistic = f1_score(y_test, y_pred_logistic)
mcc_logistic = matthews_corrcoef(y_test, y_pred_logistic)

# Display results
print("Logistic Regression Performance")
print("--------------------------------")
print("Accuracy :", round(accuracy_logistic, 4))
print("AUC      :", round(auc_logistic, 4))
print("Precision:", round(precision_logistic, 4))
print("Recall   :", round(recall_logistic, 4))
print("F1 Score :", round(f1_logistic, 4))
print("MCC      :", round(mcc_logistic, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_logistic))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_logistic,
    target_names=["Benign", "Malignant"]
))

Logistic Regression Performance
--------------------------------
Accuracy : 0.9649
AUC      : 0.996
Precision: 0.975
Recall   : 0.9286
F1 Score : 0.9512
MCC      : 0.9245

Confusion Matrix:
[[71  1]
 [ 3 39]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.96      0.99      0.97        72
   Malignant       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



## Model 2: Decision Tree Classifier

Decision Tree is a supervised machine learning algorithm that performs classification by learning decision rules from the features in the training data.

For this dataset, the Decision Tree Classifier is used to classify breast tumors as:

- **0 = Benign**
- **1 = Malignant**

The model is trained using the training dataset and evaluated on the test dataset using the following metrics:

- Accuracy
- AUC
- Precision
- Recall
- F1 Score
- Matthews Correlation Coefficient (MCC)

In [85]:
from sklearn.tree import DecisionTreeClassifier

# Create and train the Decision Tree model
decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(X_train, y_train)

# Predictions
y_pred_dt = decision_tree_model.predict(X_test)
y_prob_dt = decision_tree_model.predict_proba(X_test)[:, 1]

# Calculate performance metrics
accuracy_dt = accuracy_score(y_test, y_pred_dt)
auc_dt = roc_auc_score(y_test, y_prob_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
mcc_dt = matthews_corrcoef(y_test, y_pred_dt)

# Display results
print("Decision Tree Performance")
print("-------------------------")
print("Accuracy :", round(accuracy_dt, 4))
print("AUC      :", round(auc_dt, 4))
print("Precision:", round(precision_dt, 4))
print("Recall   :", round(recall_dt, 4))
print("F1 Score :", round(f1_dt, 4))
print("MCC      :", round(mcc_dt, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_dt,
    target_names=["Benign", "Malignant"]
))

Decision Tree Performance
-------------------------
Accuracy : 0.9298
AUC      : 0.9246
Precision: 0.9048
Recall   : 0.9048
F1 Score : 0.9048
MCC      : 0.8492

Confusion Matrix:
[[68  4]
 [ 4 38]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.94      0.94      0.94        72
   Malignant       0.90      0.90      0.90        42

    accuracy                           0.93       114
   macro avg       0.92      0.92      0.92       114
weighted avg       0.93      0.93      0.93       114



## Model 3: K-Nearest Neighbors (KNN)

K-Nearest Neighbors (KNN) is a supervised machine learning algorithm that classifies a data point based on the classes of its nearest neighboring data points.

For this dataset, KNN is used to classify breast tumors as:

- **0 = Benign**
- **1 = Malignant**

Since KNN is a distance-based algorithm, the standardized training and testing data are used.

The model is evaluated using the following metrics:

- Accuracy
- AUC
- Precision
- Recall
- F1 Score
- Matthews Correlation Coefficient (MCC)

In [87]:
from sklearn.neighbors import KNeighborsClassifier

# Create and train the KNN model
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_knn = knn_model.predict(X_test_scaled)
y_prob_knn = knn_model.predict_proba(X_test_scaled)[:, 1]

# Calculate performance metrics
accuracy_knn = accuracy_score(y_test, y_pred_knn)
auc_knn = roc_auc_score(y_test, y_prob_knn)
precision_knn = precision_score(y_test, y_pred_knn)
recall_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)
mcc_knn = matthews_corrcoef(y_test, y_pred_knn)

# Display results
print("KNN Performance")
print("---------------")
print("Accuracy :", round(accuracy_knn, 4))
print("AUC      :", round(auc_knn, 4))
print("Precision:", round(precision_knn, 4))
print("Recall   :", round(recall_knn, 4))
print("F1 Score :", round(f1_knn, 4))
print("MCC      :", round(mcc_knn, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_knn,
    target_names=["Benign", "Malignant"]
))

KNN Performance
---------------
Accuracy : 0.9561
AUC      : 0.9823
Precision: 0.9744
Recall   : 0.9048
F1 Score : 0.9383
MCC      : 0.9058

Confusion Matrix:
[[71  1]
 [ 4 38]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.95      0.99      0.97        72
   Malignant       0.97      0.90      0.94        42

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



## Model 4: Naive Bayes Classifier

Naive Bayes is a supervised machine learning classification algorithm based on Bayes' theorem. Gaussian Naive Bayes is suitable for datasets containing continuous numerical features.

For this dataset, the Gaussian Naive Bayes model is used to classify breast tumors as:

- **0 = Benign**
- **1 = Malignant**

The model is trained using the training dataset and evaluated on the test dataset using the following metrics:

- Accuracy
- AUC
- Precision
- Recall
- F1 Score
- Matthews Correlation Coefficient (MCC)

In [89]:
from sklearn.naive_bayes import GaussianNB

# Create and train the Naive Bayes model
naive_bayes_model = GaussianNB()
naive_bayes_model.fit(X_train, y_train)

# Predictions
y_pred_nb = naive_bayes_model.predict(X_test)
y_prob_nb = naive_bayes_model.predict_proba(X_test)[:, 1]

# Calculate performance metrics
accuracy_nb = accuracy_score(y_test, y_pred_nb)
auc_nb = roc_auc_score(y_test, y_prob_nb)
precision_nb = precision_score(y_test, y_pred_nb)
recall_nb = recall_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)
mcc_nb = matthews_corrcoef(y_test, y_pred_nb)

# Display results
print("Naive Bayes Performance")
print("-----------------------")
print("Accuracy :", round(accuracy_nb, 4))
print("AUC      :", round(auc_nb, 4))
print("Precision:", round(precision_nb, 4))
print("Recall   :", round(recall_nb, 4))
print("F1 Score :", round(f1_nb, 4))
print("MCC      :", round(mcc_nb, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_nb,
    target_names=["Benign", "Malignant"]
))

Naive Bayes Performance
-----------------------
Accuracy : 0.9386
AUC      : 0.9934
Precision: 1.0
Recall   : 0.8333
F1 Score : 0.9091
MCC      : 0.8715

Confusion Matrix:
[[72  0]
 [ 7 35]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.91      1.00      0.95        72
   Malignant       1.00      0.83      0.91        42

    accuracy                           0.94       114
   macro avg       0.96      0.92      0.93       114
weighted avg       0.94      0.94      0.94       114



## Model 5: Support Vector Machine (SVM)

Support Vector Machine (SVM) is a supervised machine learning algorithm used for classification. It identifies an optimal decision boundary that separates different classes in the feature space.

For this dataset, SVM is used to classify breast tumors as:

- **0 = Benign**
- **1 = Malignant**

Since SVM is sensitive to differences in feature scales, the standardized training and testing data are used.

The model is evaluated using the following metrics:

- Accuracy
- AUC
- Precision
- Recall
- F1 Score
- Matthews Correlation Coefficient (MCC)

In [91]:
from sklearn.svm import SVC

# Create and train the SVM model
svm_model = SVC(probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_svm = svm_model.predict(X_test_scaled)
y_prob_svm = svm_model.predict_proba(X_test_scaled)[:, 1]

# Calculate performance metrics
accuracy_svm = accuracy_score(y_test, y_pred_svm)
auc_svm = roc_auc_score(y_test, y_prob_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)
mcc_svm = matthews_corrcoef(y_test, y_pred_svm)

# Display results
print("SVM Performance")
print("---------------")
print("Accuracy :", round(accuracy_svm, 4))
print("AUC      :", round(auc_svm, 4))
print("Precision:", round(precision_svm, 4))
print("Recall   :", round(recall_svm, 4))
print("F1 Score :", round(f1_svm, 4))
print("MCC      :", round(mcc_svm, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_svm,
    target_names=["Benign", "Malignant"]
))

SVM Performance
---------------
Accuracy : 0.9737
AUC      : 0.9947
Precision: 1.0
Recall   : 0.9286
F1 Score : 0.963
MCC      : 0.9442

Confusion Matrix:
[[72  0]
 [ 3 39]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.96      1.00      0.98        72
   Malignant       1.00      0.93      0.96        42

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114



/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


## Model Performance Comparison

The performance of all five classification models is compared using the six evaluation metrics required for this assignment:

- Accuracy
- AUC
- Precision
- Recall
- F1 Score
- Matthews Correlation Coefficient (MCC)

The comparison helps identify the model that provides the best overall classification performance on the Breast Cancer Wisconsin dataset.

In [111]:
# Create comparison table
comparison_df = pd.DataFrame({
    "ML Model": [
        "Logistic Regression",
        "Decision Tree",
        "KNN",
        "Naive Bayes",
        "Support Vector Machine"
    ],
    "Accuracy": [
        accuracy_logistic,
        accuracy_dt,
        accuracy_knn,
        accuracy_nb,
        accuracy_svm
    ],
    "AUC": [
        auc_logistic,
        auc_dt,
        auc_knn,
        auc_nb,
        auc_svm
    ],
    "Precision": [
        precision_logistic,
        precision_dt,
        precision_knn,
        precision_nb,
        precision_svm
    ],
    "Recall": [
        recall_logistic,
        recall_dt,
        recall_knn,
        recall_nb,
        recall_svm
    ],
    "F1 Score": [
        f1_logistic,
        f1_dt,
        f1_knn,
        f1_nb,
        f1_svm
    ],
    "MCC": [
        mcc_logistic,
        mcc_dt,
        mcc_knn,
        mcc_nb,
        mcc_svm
    ]
})

# Round values for readability
comparison_df = comparison_df.round(4)

# Display the comparison table
comparison_df

,ML Model,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Logistic Regression,0.9649,0.9960,0.9750,0.9286,0.9512,0.9245
1,Decision Tree,0.9298,0.9246,0.9048,0.9048,0.9048,0.8492
2,KNN,0.9561,0.9823,0.9744,0.9048,0.9383,0.9058
3,Naive Bayes,0.9386,0.9934,1.0000,0.8333,0.9091,0.8715
4,Support Vector Machine,0.9737,0.9947,1.0000,0.9286,0.9630,0.9442


## Observations and Model Comparison

The five classification models were evaluated using Accuracy, AUC, Precision, Recall, F1 Score, and Matthews Correlation Coefficient (MCC).

### 1. Logistic Regression
Logistic Regression achieved an Accuracy of **0.9649**, AUC of **0.9960**, Precision of **0.9750**, Recall of **0.9286**, F1 Score of **0.9512**, and MCC of **0.9245**. It achieved the highest AUC among all the models, indicating excellent ability to distinguish between benign and malignant cases.

### 2. Decision Tree
Decision Tree achieved an Accuracy of **0.9298**, AUC of **0.9246**, Precision of **0.9048**, Recall of **0.9048**, F1 Score of **0.9048**, and MCC of **0.8492**. Although the model performed reasonably well, its overall performance was lower than the other models.

### 3. K-Nearest Neighbors (KNN)
KNN achieved an Accuracy of **0.9561**, AUC of **0.9823**, Precision of **0.9744**, Recall of **0.9048**, F1 Score of **0.9383**, and MCC of **0.9058**. The model demonstrated strong classification performance after feature standardization.

### 4. Naive Bayes
Naive Bayes achieved an Accuracy of **0.9386**, AUC of **0.9934**, Precision of **1.0000**, Recall of **0.8333**, F1 Score of **0.9091**, and MCC of **0.8715**. The model achieved perfect precision but had the lowest recall among the five models, indicating that some malignant cases were not correctly identified.

### 5. Support Vector Machine (SVM)
SVM achieved an Accuracy of **0.9737**, AUC of **0.9947**, Precision of **1.0000**, Recall of **0.9286**, F1 Score of **0.9630**, and MCC of **0.9442**. SVM achieved the highest Accuracy, F1 Score, and MCC, along with perfect Precision and strong Recall.

### Overall Observation

Based on the overall comparison, **Support Vector Machine (SVM) is the best-performing model**. SVM achieved the highest Accuracy (**0.9737**), F1 Score (**0.9630**), and MCC (**0.9442**), while also achieving perfect Precision (**1.0000**) and a very high AUC (**0.9947**). Although Logistic Regression achieved a slightly higher AUC of **0.9960**, SVM provided the strongest overall performance across the evaluation metrics. Therefore, SVM is selected as the best model for this classification problem.

## Saving Trained Models and Test Data

The trained machine learning models are saved so that they can be loaded directly by the Streamlit application without retraining.

The StandardScaler used during preprocessing is also saved because Logistic Regression, KNN, and SVM require standardized input features.

The test dataset is saved separately as `test_data.csv`. It contains the input features along with the target variable `diagnosis` and will be used to evaluate the trained models in the Streamlit application.


In [121]:
import joblib
import pandas as pd

# Save all trained models
joblib.dump(logistic_model, "logistic_model.pkl")
joblib.dump(decision_tree_model, "decision_tree_model.pkl")
joblib.dump(knn_model, "knn_model.pkl")
joblib.dump(naive_bayes_model, "naive_bayes_model.pkl")
joblib.dump(svm_model, "svm_model.pkl")

# Save the scaler
joblib.dump(scaler, "scaler.pkl")

# Create test dataset
test_data = X_test.copy()
test_data["diagnosis"] = y_test.values

# Save test dataset
test_data.to_csv("test_data.csv", index=False)

print("Files saved successfully:")
print("1. logistic_model.pkl")
print("2. decision_tree_model.pkl")
print("3. knn_model.pkl")
print("4. naive_bayes_model.pkl")
print("5. svm_model.pkl")
print("6. scaler.pkl")
print("7. test_data.csv")

Files saved successfully:
1. logistic_model.pkl
2. decision_tree_model.pkl
3. knn_model.pkl
4. naive_bayes_model.pkl
5. svm_model.pkl
6. scaler.pkl
7. test_data.csv
